# Predict Bitcoin price movement using Linear Regression (Databricks)

This notebook builds a simple Spark ML pipeline to predict next-day Bitcoin price movement using linear regression.

What it demonstrates:
- Spark data prep and time-series feature engineering
- Features: MA7, RSI(14), volume, 1-day return
- Spark ML LinearRegression in a Pipeline
- MLflow tracking (params, metrics, model artifact)

Input data requirements (CSV):
- A date column and numeric columns for close and volume
- Common headers are supported (Date/Open/High/Low/Close/Volume), case-insensitive

Ingestion modes:
- CSV from DBFS
- Download BTC-USD from Yahoo Finance via yfinance
- Synthetic OHLCV-like data (fallback)


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark

import pandas as pd
import subprocess
import datetime as dt
import math
import random

## Data ingestion

Choose one of:
- CSV in DBFS
- BTC-USD from Yahoo Finance (yfinance)

CSV examples:
- `dbfs:/FileStore/bitcoin/btc.csv`
- `dbfs:/mnt/<mount>/btc.csv`

If ingestion fails, synthetic data is used as a fallback.

In [ ]:
dbutils.widgets.text("source_path", "")
dbutils.widgets.dropdown("ingest_mode", "csv", ["csv", "yfinance_btcusd", "synthetic"], "Ingest mode")
dbutils.widgets.text("yf_period", "5y")
dbutils.widgets.text("yf_start", "")
dbutils.widgets.text("yf_end", "")
dbutils.widgets.text("mlflow_experiment", "/Shared/bitcoin_lr_linear_regression")
dbutils.widgets.text("predictions_table", "")

source_path = dbutils.widgets.get("source_path").strip()
ingest_mode = dbutils.widgets.get("ingest_mode").strip()
yf_period = dbutils.widgets.get("yf_period").strip()
yf_start = dbutils.widgets.get("yf_start").strip()
yf_end = dbutils.widgets.get("yf_end").strip()
experiment_path = dbutils.widgets.get("mlflow_experiment").strip()
predictions_table = dbutils.widgets.get("predictions_table").strip()

def _dbfs_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def _install_yfinance_if_needed():
    try:
        import yfinance as yf
        return yf
    except Exception:
        subprocess.check_call(["python", "-m", "pip", "install", "-q", "yfinance"]) 
        import yfinance as yf
        return yf

def _parse_date(s: str):
    if not s:
        return None
    return dt.datetime.strptime(s, "%Y-%m-%d").date()

def _read_yfinance_btcusd():
    yf = _install_yfinance_if_needed()
    start = _parse_date(yf_start)
    end = _parse_date(yf_end)
    if start is None:
        pdf = yf.download("BTC-USD", period=yf_period or "5y", interval="1d", auto_adjust=False, progress=False)
    else:
        pdf = yf.download("BTC-USD", start=start, end=end, interval="1d", auto_adjust=False, progress=False)
    if pdf is None or len(pdf) == 0:
        raise ValueError("No rows returned from yfinance for BTC-USD")
    pdf = pdf.reset_index()
    rename = {"Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    pdf = pdf.rename(columns=rename)
    pdf["date"] = pd.to_datetime(pdf["date"]).dt.date
    for c in ["open", "high", "low", "close", "volume"]:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
    pdf = pdf.dropna(subset=["date", "close", "volume"])
    return spark.createDataFrame(pdf[["date", "open", "high", "low", "close", "volume"]])

def _normalize_columns(df):
    cols = {c.lower(): c for c in df.columns}
    rename_map = {}
    for canonical, candidates in {
        "date": ["date", "timestamp", "time"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close", "adjclose"],
        "volume": ["volume", "vol"]
    }.items():
        for cand in candidates:
            if cand in cols:
                rename_map[cols[cand]] = canonical
                break

    for old, new in rename_map.items():
        df = df.withColumnRenamed(old, new)
    return df

raw = None
if ingest_mode == "yfinance_btcusd":
    try:
        raw = _read_yfinance_btcusd()
    except Exception:
        raw = None

if raw is None and ingest_mode in ["csv", "yfinance_btcusd"] and source_path and _dbfs_exists(source_path):
    raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
    )
    raw = _normalize_columns(raw)

if raw is None:
    random.seed(42)
    start = dt.date(2020, 1, 1)
    days = 900
    price = 10000.0
    rows = []
    for i in range(days):
        date = start + dt.timedelta(days=i)
        drift = 0.0004
        shock = random.gauss(0.0, 0.02)
        ret = drift + shock
        close = price * math.exp(ret)
        high = max(price, close) * (1.0 + abs(random.gauss(0.0, 0.01)))
        low = min(price, close) * (1.0 - abs(random.gauss(0.0, 0.01)))
        open_ = price
        volume = int(1_000_000 + abs(random.gauss(0, 200_000)))
        rows.append((date, float(open_), float(high), float(low), float(close), int(volume)))
        price = close

    schema = StructType([
        StructField("date", DateType(), False),
        StructField("open", DoubleType(), False),
        StructField("high", DoubleType(), False),
        StructField("low", DoubleType(), False),
        StructField("close", DoubleType(), False),
        StructField("volume", LongType(), False)
    ])
    raw = spark.createDataFrame(rows, schema=schema)

raw = (
    raw
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("close", F.col("close").cast("double"))
    .withColumn("volume", F.col("volume").cast("double"))
    .filter(F.col("date").isNotNull())
    .filter(F.col("close").isNotNull())
    .filter(F.col("volume").isNotNull())
)

raw.orderBy("date").display()

## Feature engineering (MA7, RSI(14), volume)

Target:
- `label_return_1d_ahead`: next-day return based on close price

Features:
- `ma7_close`: 7-day moving average of close
- `rsi14`: RSI computed from 14-day average gains/losses
- `volume`
- `return_1d`: current 1-day return

In [ ]:
w_order = Window.orderBy("date")
w_ma7 = w_order.rowsBetween(-6, 0)
w_rsi14 = w_order.rowsBetween(-13, 0)

df = (
    raw.orderBy("date")
    .withColumn("prev_close", F.lag("close", 1).over(w_order))
    .withColumn("return_1d", (F.col("close") - F.col("prev_close")) / F.col("prev_close"))
    .withColumn("ma7_close", F.avg("close").over(w_ma7))
)

df = (
    df
    .withColumn("delta", F.col("close") - F.col("prev_close"))
    .withColumn("gain", F.when(F.col("delta") > 0, F.col("delta")).otherwise(F.lit(0.0)))
    .withColumn("loss", F.when(F.col("delta") < 0, -F.col("delta")).otherwise(F.lit(0.0)))
    .withColumn("avg_gain_14", F.avg("gain").over(w_rsi14))
    .withColumn("avg_loss_14", F.avg("loss").over(w_rsi14))
    .withColumn(
        "rsi14",
        F.when(F.col("avg_loss_14") == 0, F.lit(100.0))
        .otherwise(100.0 - (100.0 / (1.0 + (F.col("avg_gain_14") / F.col("avg_loss_14")))))
    )
)

df = df.withColumn("label_return_1d_ahead", F.lead("return_1d", 1).over(w_order))

dataset = (
    df
    .select("date", "close", "volume", "return_1d", "ma7_close", "rsi14", "label_return_1d_ahead")
    .filter(F.col("return_1d").isNotNull())
    .filter(F.col("ma7_close").isNotNull())
    .filter(F.col("rsi14").isNotNull())
    .filter(F.col("label_return_1d_ahead").isNotNull())
)

dataset.orderBy("date").display()

## Train/test split (time-based)

Splits by date order so the test set is strictly after the training period.

In [ ]:
n = dataset.count()
train_n = int(n * 0.8)

dataset_idx = dataset.withColumn("row_num", F.row_number().over(w_order))
train_df = dataset_idx.filter(F.col("row_num") <= F.lit(train_n)).drop("row_num")
test_df = dataset_idx.filter(F.col("row_num") > F.lit(train_n)).drop("row_num")

train_df.select(F.min("date"), F.max("date"), F.count("*").alias("rows")).display()
test_df.select(F.min("date"), F.max("date"), F.count("*").alias("rows")).display()

## Spark ML pipeline + MLflow tracking

Model:
- LinearRegression predicting `label_return_1d_ahead`

Metrics:
- RMSE and R2 for regression fit
- Direction accuracy: whether sign(prediction) matches sign(label)

In [ ]:
label_col = "label_return_1d_ahead"
feature_cols = ["ma7_close", "rsi14", "volume", "return_1d"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
lr = LinearRegression(
    featuresCol="features",
    labelCol=label_col,
    regParam=0.0,
    elasticNetParam=0.0,
    maxIter=100
)

pipeline = Pipeline(stages=[assembler, scaler, lr])

mlflow.set_experiment(experiment_path)

with mlflow.start_run(run_name="bitcoin_lr_spark"):
    mlflow.log_param("label", label_col)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("regParam", lr.getRegParam())
    mlflow.log_param("elasticNetParam", lr.getElasticNetParam())
    mlflow.log_param("maxIter", lr.getMaxIter())
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())

    model = pipeline.fit(train_df)
    preds = model.transform(test_df)

    rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse").evaluate(preds)
    r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2").evaluate(preds)

    direction_acc = (
        preds
        .select(
            F.when(F.col("prediction") >= 0, 1).otherwise(0).alias("pred_up"),
            F.when(F.col(label_col) >= 0, 1).otherwise(0).alias("label_up")
        )
        .withColumn("correct", (F.col("pred_up") == F.col("label_up")).cast("double"))
        .agg(F.avg("correct").alias("direction_accuracy"))
        .first()[0]
    )

    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))
    mlflow.log_metric("direction_accuracy", float(direction_acc))

    mlflow.spark.log_model(model, artifact_path="model")

preds.select("date", "close", "volume", "ma7_close", "rsi14", "return_1d", label_col, "prediction").orderBy(F.desc("date")).display()

## Optional: save predictions to a table

Set `predictions_table` widget to a table name to persist predictions.

In [ ]:
if predictions_table:
    (
        preds
        .select(
            "date",
            "close",
            "volume",
            "ma7_close",
            "rsi14",
            "return_1d",
            F.col(label_col).alias("label"),
            "prediction"
        )
        .write
        .mode("overwrite")
        .saveAsTable(predictions_table)
    )
    spark.table(predictions_table).orderBy(F.desc("date")).display()